# Training an "Ahoj Hugo" microWakeWord model for the HA Voice PE

Produces `ahoj_hugo.tflite` + `ahoj_hugo.json`, ready for havoice
(`cargo run --example wake_word -- verify ahoj_hugo.tflite --wake-word "Ahoj Hugo"`).

**Before starting:**

1. On your Mac, run `python3 training/generate_samples.py --count 1000 --out samples/ahoj_hugo`
   (see `training/generate_samples.py` for the edge-tts / `say` / real-recording options).
2. In Colab: *Runtime -> Change runtime type -> GPU* (T4 is fine). Training takes a few hours.
3. Have the `samples/ahoj_hugo.zip` produced above ready to upload in the next step
   (or put it on Google Drive and mount the drive instead of uploading).

This wraps [kahrendt/microWakeWord](https://github.com/kahrendt/microWakeWord); the model
can still be hard to trigger or false-trigger - expect to iterate on the settings marked
below. The negative datasets mix licenses (AudioSet, FMA), so treat the trained model as
**non-commercial, personal use**.

In [ ]:
# Installs microWakeWord. If Colab prompts to restart the session afterwards, do it.
import platform

if platform.system() == "Darwin":
    # `pymicro-features` is installed from a fork to support building on macOS
    !pip install 'git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version'

# `audio-metadata` is installed from a fork to unpin `attrs` from a version that breaks Jupyter
!pip install 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'

!git clone https://github.com/kahrendt/microWakeWord
!pip install -e ./microWakeWord

In [ ]:
# Upload the zip produced by training/generate_samples.py and unpack it.
from google.colab import files

uploaded = files.upload()
zip_name = next(iter(uploaded))

!unzip -o -q {zip_name} -d generated_samples
!ls generated_samples | head -5
!ls generated_samples | wc -l

In [ ]:
# Play one sample to check you uploaded the right thing.
import glob, random

from IPython.display import Audio

sample = random.choice(glob.glob("generated_samples/*.wav"))
print(sample)
Audio(sample)

In [ ]:
# Downloads audio data for augmentation. This can be slow!
# Borrowed from openWakeWord's automatic_model_training.ipynb, March 4, 2024.
# The data mixes licenses - models trained with it are for non-commercial use.

import datasets
import os
import scipy

import numpy as np

from pathlib import Path
from tqdm import tqdm

## MIT RIR data

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    rir_dataset = datasets.load_dataset(
        "davidscripka/MIT_environmental_impulse_responses",
        split="train",
        streaming=True,
    )
    for row in tqdm(rir_dataset):
        name = row['audio']['path'].split('/')[-1]
        scipy.io.wavfile.write(
            os.path.join(output_dir, name), 16000,
            (row['audio']['array']*32767).astype(np.int16),
        )

## AudioSet (one part; use the full dataset from
## https://huggingface.co/datasets/agkphysics/AudioSet for better models)

if not os.path.exists("audioset"):
    os.mkdir("audioset")

    fname = "bal_train09.tar"
    out_dir = f"audioset/{fname}"
    link = ("https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/"
            + fname)
    !wget -O {out_dir} {link}
    !cd audioset && tar -xf bal_train09.tar

    output_dir = "./audioset_16k"
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)

    audioset_dataset = datasets.Dataset.from_dict(
        {"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
    audioset_dataset = audioset_dataset.cast_column(
        "audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(audioset_dataset):
        name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
        scipy.io.wavfile.write(
            os.path.join(output_dir, name), 16000,
            (row['audio']['array']*32767).astype(np.int16),
        )

## Free Music Archive (third-party mchl914 extra small set)

output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    fname = "fma_xs.zip"
    link = "https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/" + fname
    out_dir = f"fma/{fname}"
    !wget -O {out_dir} {link}
    !cd {output_dir} && unzip -q {fname}

    output_dir = "./fma_16k"
    if not os.path.exists(output_dir):
        os.mkdir(output_dir)

    fma_dataset = datasets.Dataset.from_dict(
        {"audio": [str(i) for i in Path("fma/fma_small").glob("**/*.mp3")]})
    fma_dataset = fma_dataset.cast_column(
        "audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(fma_dataset):
        name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
        scipy.io.wavfile.write(
            os.path.join(output_dir, name), 16000,
            (row['audio']['array']*32767).astype(np.int16),
        )

In [ ]:
# Sets up the augmentations. Experiment with these to improve the model.

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

clips = Clips(input_directory='generated_samples',
              file_pattern='*.wav',
              max_clip_duration_s=None,
              remove_silence=False,
              random_split_seed=10,
              split_count=0.1,
              )
augmenter = Augmentation(augmentation_duration_s=3.2,
                         augmentation_probabilities = {
                                "SevenBandParametricEQ": 0.1,
                                "TanhDistortion": 0.1,
                                "PitchShift": 0.1,
                                "BandStopFilter": 0.1,
                                "AddColorNoise": 0.1,
                                "AddBackgroundNoise": 0.75,
                                "Gain": 1.0,
                                "RIR": 0.5,
                            },
                         impulse_paths = ['mit_rirs'],
                         background_paths = ['fma_16k', 'audioset_16k'],
                         background_min_snr_db = -5,
                         background_max_snr_db = 10,
                         min_jitter_s = 0.195,
                         max_jitter_s = 0.205,
                         )

In [ ]:
# Augment a random clip and play it back to verify it sounds
# like a real room recording.

from IPython.display import Audio
from microwakeword.audio.audio_utils import save_clip

random_clip = clips.get_random_clip()
augmented_clip = augmenter.augment_clip(random_clip)
save_clip(augmented_clip, 'augmented_clip.wav')

Audio("augmented_clip.wav", autoplay=True)

In [ ]:
# Augment samples and save the training, validation, and testing sets.

import os
from mmap_ninja.ragged import RaggedMmap

output_dir = 'generated_augmented_features'

if not os.path.exists(output_dir):
    os.mkdir(output_dir)

splits = ["training", "validation", "testing"]
for split in splits:
    out_dir = os.path.join(output_dir, split)
    if not os.path.exists(out_dir):
        os.mkdir(out_dir)

    split_name = "train"
    repetition = 2

    spectrograms = SpectrogramGeneration(clips=clips,
                                        augmenter=augmenter,
                                        slide_frames=10,
                                        step_ms=10,
                                        )
    if split == "validation":
        split_name = "validation"
        repetition = 1
    elif split == "testing":
        split_name = "test"
        repetition = 1
        spectrograms = SpectrogramGeneration(clips=clips,
                                            augmenter=augmenter,
                                            slide_frames=1,
                                            step_ms=10,
                                            )

    RaggedMmap.from_generator(
        out_dir=os.path.join(out_dir, 'wakeword_mmap'),
        sample_generator=spectrograms.spectrogram_generator(
            split=split_name, repeat=repetition),
        batch_size=1,
    )

In [ ]:
# Downloads pre-generated negative spectrogram features. This can be slow!

output_dir = './negative_datasets'
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    link_root = "https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/"
    filenames = ['dinner_party.zip', 'dinner_party_eval.zip',
                 'no_speech.zip', 'speech.zip']
    for fname in filenames:
        link = link_root + fname

        zip_path = f"negative_datasets/{fname}"
        !wget -O {zip_path} {link}
        !unzip -q {zip_path} -d {output_dir}

In [ ]:
# Training configuration, tuned for a Czech two-word wake word.
# Iterate here when the model triggers poorly: more steps, different
# sampling/penalty weights, SpecAugment, or more (and more varied)
# positive samples.

import yaml

config = {}

config["window_step_ms"] = 10
config["train_dir"] = "trained_models/wakeword"

config["features"] = [
    {
        "features_dir": "generated_augmented_features",
        "sampling_weight": 2.0,
        "penalty_weight": 1.0,
        "truth": True,
        "truncation_strategy": "truncate_start",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/speech",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/dinner_party",
        "sampling_weight": 10.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/no_speech",
        "sampling_weight": 5.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "random",
        "type": "mmap",
    },
    {
        "features_dir": "negative_datasets/dinner_party_eval",
        "sampling_weight": 0.0,
        "penalty_weight": 1.0,
        "truth": False,
        "truncation_strategy": "split",
        "type": "mmap",
    },
]

config["training_steps"] = [15000, 5000]
config["positive_class_weight"] = [1, 1]
config["negative_class_weight"] = [20, 20]
config["learning_rates"] = [0.001, 0.0001]
config["batch_size"] = 128

config["time_mask_max_size"] = [0, 10]
config["time_mask_count"] = [0, 2]
config["freq_mask_max_size"] = [0, 4]
config["freq_mask_count"] = [0, 1]

config["eval_step_interval"] = 500
config["clip_duration_ms"] = 1500

config["target_minimization"] = 0.9
config["minimization_metric"] = None
config["maximization_metric"] = "average_viable_recall"

with open("training_parameters.yaml", "w") as file:
    yaml.dump(config, file)

In [ ]:
# Trains, then quantizes and converts to the streaming model the device
# runs. Resumes if stopped. On Colab the mini-batch output may not appear
# for minutes! Consider --train 0 to only convert and test the best
# weights of a previous run.

!python -m microwakeword.model_train_eval \
--training_config='training_parameters.yaml' \
--train 1 \
--restore_checkpoint 1 \
--test_tf_nonstreaming 0 \
--test_tflite_nonstreaming 0 \
--test_tflite_nonstreaming_quantized 0 \
--test_tflite_streaming 0 \
--test_tflite_streaming_quantized 1 \
--use_weights "best_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block  "1, 1, 1, 1" \
--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

In [ ]:
# Write the v2 manifest havoice and ESPHome expect, and package the
# result. Lower probability_cutoff if the device misses the wake word;
# raise it if it wakes up on its own. tensor_arena_size is a safe upper
# bound for this architecture (the official models of the same shape
# use ~26 kB).

import json
import shutil

from google.colab import files

tflite_src = ("trained_models/wakeword/"
              "tflite_stream_state_internal_quant/"
              "stream_state_internal_quant.tflite")
shutil.copy(tflite_src, "ahoj_hugo.tflite")

manifest = {
    "type": "micro",
    "wake_word": "Ahoj Hugo",
    "author": "skoky",
    "website": "https://github.com/skoky/havoice",
    "model": "ahoj_hugo.tflite",
    "trained_languages": ["cs"],
    "version": 2,
    "micro": {
        "probability_cutoff": 0.97,
        "feature_step_size": 10,
        "sliding_window_size": 5,
        "tensor_arena_size": 30000,
        "minimum_esphome_version": "2024.7.0",
    },
}
with open("ahoj_hugo.json", "w") as file:
    json.dump(manifest, file, indent=2)

!zip -q ahoj_hugo_model.zip ahoj_hugo.tflite ahoj_hugo.json
files.download("ahoj_hugo_model.zip")

## Next steps with havoice

```sh
unzip ahoj_hugo_model.zip
cargo run --example wake_word -- verify ahoj_hugo.tflite --wake-word "Ahoj Hugo"
```

All checks must pass (int8 `[1, stride, 40]` input, uint8 `[1, 1]` output, only the 20
ops the device interpreter has, arena within 200 kB). Then flash it:

```sh
cargo run --example wake_word -- install ahoj_hugo.tflite --wake-word "Ahoj Hugo" --flash
```

Test it in a real room. If it misses wake-ups, lower `probability_cutoff` (0.95, 0.9...)
in `ahoj_hugo.json`; if it false-triggers, raise it or retrain with more negative
Czech speech and a higher cutoff. Regenerate samples with `--edge`/`--say` and your own
recordings (`--extra-wavs`) for a more robust model.